# MedAgentBench GRPO Training with Unsloth + TRL

Fine-tune **Qwen3-1.7B** on the MedAgentBench clinical decision-making benchmark using:
- **Unsloth** for 2-4x faster LoRA fine-tuning
- **TRL GRPOTrainer** for RL with environment tool-use interaction
- **Mock FHIR** backend (no live server needed)

Uses the OpenEnv environment deployed at [HuggingFace Spaces](https://huggingface.co/spaces/amantra/medagentbench_env).

**Runtime:** Colab T4/A100 GPU required

## 1. Install Dependencies & Clone Environment

In [1]:
# Install Unsloth + TRL
!pip install -q unsloth trl datasets pydantic

# Clone our OpenEnv environment from HuggingFace Spaces
!git clone https://huggingface.co/spaces/amantra/medagentbench_env ./medagentbench_env_repo

# Verify GPU
!nvidia-smi

fatal: destination path './medagentbench_env_repo' already exists and is not an empty directory.
Sun Mar  8 15:35:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   25C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |

## 2. Load Model with Unsloth (4-bit LoRA)

In [5]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen3-1.7B"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_NAME}")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: Qwen/Qwen3-1.7B
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## 3. Load Environment & Data from OpenEnv

Import `train.py` directly from the cloned repo -- includes MockFHIR backend,
shaped reward function, and all FHIR tool definitions.

In [6]:
import importlib.util
from pathlib import Path

# Load train.py directly (bypasses openenv dependency in package __init__)
spec = importlib.util.spec_from_file_location("train", "./medagentbench_env_repo/train.py")
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

# Pull out what we need
MedAgentTrainEnv = train_mod.MedAgentTrainEnv
reward_func = train_mod.reward_func
build_dataset = train_mod.build_dataset

DATA_DIR = Path("./medagentbench_env_repo/data")

# Pre-load shared resources (FHIR cache + tasks)
train_mod._get_mock_fhir()
tasks = train_mod._get_tasks()
print(f"Environment loaded: {len(tasks)} tasks")

# Sanity check: run one episode
env = MedAgentTrainEnv()
instruction = env.reset()
print(f"Task: {env._task['id']}")
print(f"Instruction: {instruction[:120]}...")
result = env.finish(["test"])
print(f"Result: {result}")

Environment loaded: 90 tasks
Task: task3_1
Instruction: I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context...
Result: Task completed. Reward: 0.138


## 4. Build Training Dataset

In [ ]:
from datasets import Dataset

dataset = build_dataset(DATA_DIR)
print(f"Training dataset: {len(dataset)} tasks")
print(f"Example prompt: {[m['role'] for m in dataset[0]['prompt']]}")

## 5. Train with GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./medagent_grpo_output"

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_completion_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=2,
    bf16=True,
    chat_template_kwargs={"enable_thinking": False},
    save_steps=50,
    save_total_limit=2,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    train_dataset=dataset,
    environment_factory=MedAgentTrainEnv,
    processing_class=tokenizer,
    args=grpo_config,
)

print("Starting GRPO training...")
trainer.train()

## 6. Save Model

In [ ]:
# Save LoRA adapter + tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Optional: merge LoRA weights and save full model
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained(f"{OUTPUT_DIR}_merged")

## 7. Quick Evaluation

Run the trained model on a few tasks to compare against baseline.

In [ ]:
import json, re
from urllib.parse import urlparse, parse_qs

# Reset task counter
train_mod._TASK_INDEX = 0

SYSTEM_PROMPT = (
    "You are an expert medical AI agent. "
    "Use the available FHIR tools to complete the clinical task. "
    "Always call finish when you are done."
)

FastLanguageModel.for_inference(model)

rewards = []
num_eval = min(10, len(tasks))

for i in range(num_eval):
    env = MedAgentTrainEnv()
    instruction = env.reset()
    task_id = env._task["id"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    for step in range(8):
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        reply = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        if "FINISH(" in reply:
            m = re.search(r"FINISH\((.+)\)", reply, re.DOTALL)
            if m:
                try:
                    val = json.loads(m.group(1))
                    env.finish(val if isinstance(val, list) else [val])
                except json.JSONDecodeError:
                    env.finish([])
            else:
                env.finish([])
            break
        elif reply.strip().startswith("GET "):
            url = reply.strip()[4:].strip().split()[0]
            parsed = urlparse(url)
            resource = parsed.path.rstrip("/").split("/")[-1]
            params = {k: v[0] for k, v in parse_qs(parsed.query).items()}
            obs = env._do_get(resource, params)
        else:
            env.finish([])
            break

        if env.done:
            break

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": obs})
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

    rewards.append(env.reward)
    print(f"  {task_id}: reward={env.reward:.4f}")

avg = sum(rewards) / len(rewards) if rewards else 0
print(f"\nPost-training avg reward ({num_eval} tasks): {avg:.4f}")
print(f"Baseline avg reward (Qwen3-8B via OpenRouter): 0.2748")